In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

def criterion(out, label):
    return (label - out)**2

# (data / label)
# note floating point number *.*
data = [(1.0,3.0), (2.0,6.0), (3.0,9.0), (4.0,12.0), (5.0,15.0), (6.0,18.0)]

# requires_grad turned on
W = torch.tensor([1.0], requires_grad=True)


X,label = data[1]

Y = X*W
loss = criterion(Y,label) 
loss.backward()

## loss = (label-X*W)**2
## d(loss)/dw = 2*(label-X*W)*(-X)=2*4*(-2)

W.grad

tensor([-16.])

In [2]:
### Training with manually updating W with "Backward" ###

import torch
#from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

def criterion(out, label):
    return (label - out)**2


data = [(1.0,3.0), (2.0,6.0), (3.0,9.0), (4.0,12.0), (5.0,15.0), (6.0,18.0)]

W = torch.tensor([1.0], requires_grad=True)

lr = 0.01
temp = torch.tensor([0.0])


for epoch in range(20):
    for i, current_data in enumerate(data):

        W = temp
        W.requires_grad = True
        X, Y = current_data
        outputs = X*W
        loss = criterion(outputs, Y)
        loss.backward()
        W = W - lr* W.grad
        temp = W.detach()    # Not to deliver the grad, but just value
        print("Epoch {} - loss: {}".format(epoch, loss))


### Test the trained network ###            
for i, current_data in enumerate(data):
    X, Y = current_data 
    outputs = X*W  
    print("when x = {}, y = {}".format(X, outputs))

Epoch 0 - loss: tensor([9.], grad_fn=<PowBackward0>)
Epoch 0 - loss: tensor([34.5744], grad_fn=<PowBackward0>)
Epoch 0 - loss: tensor([65.8435], grad_fn=<PowBackward0>)
Epoch 0 - loss: tensor([78.7078], grad_fn=<PowBackward0>)
Epoch 0 - loss: tensor([56.8664], grad_fn=<PowBackward0>)
Epoch 0 - loss: tensor([20.4719], grad_fn=<PowBackward0>)
Epoch 1 - loss: tensor([0.0446], grad_fn=<PowBackward0>)
Epoch 1 - loss: tensor([0.1713], grad_fn=<PowBackward0>)
Epoch 1 - loss: tensor([0.3262], grad_fn=<PowBackward0>)
Epoch 1 - loss: tensor([0.3899], grad_fn=<PowBackward0>)
Epoch 1 - loss: tensor([0.2817], grad_fn=<PowBackward0>)
Epoch 1 - loss: tensor([0.1014], grad_fn=<PowBackward0>)
Epoch 2 - loss: tensor([0.0002], grad_fn=<PowBackward0>)
Epoch 2 - loss: tensor([0.0008], grad_fn=<PowBackward0>)
Epoch 2 - loss: tensor([0.0016], grad_fn=<PowBackward0>)
Epoch 2 - loss: tensor([0.0019], grad_fn=<PowBackward0>)
Epoch 2 - loss: tensor([0.0014], grad_fn=<PowBackward0>)
Epoch 2 - loss: tensor([0.0005

In [3]:
### Training with fancier version ###

import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim


class Net(nn.Module): ## nn.Module class is used
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(1,1,bias=False)  # in dim, out dim
    def forward(self, x):
        x = self.fc1(x)
        return x

net = Net()

print(net)
print(list(net.parameters())) # parameters are randomized

#input = torch.randn(1)
#out = net(input)

#def criterion(out, label):
#    return (label - out)**2
criterion = nn.MSELoss()


optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.5)
#optimizer = optim.Adam(net.parameters(), lr=0.005)


data = [(1.0,3.0), (2.0,6.0), (3.0,9.0), (4.0,12.0), (5.0,15.0), (6.0,18.0)]

for epoch in range(20): # 0 - 19
    for i, current_data in enumerate(data):
        X, Y = current_data
        X, Y = torch.FloatTensor([X]), torch.FloatTensor([Y])
        optimizer.zero_grad()   
        outputs = net(X)
        loss = criterion(outputs, Y)
        loss.backward()
        optimizer.step()    ## This line is equivalent to "W = W - lr* W.grad"
        print("Epoch {} - loss: {}".format(epoch, loss))

### Test the trained network ###            
for i, current_data in enumerate(data):
    X, Y = current_data
    X, Y = torch.FloatTensor([X]), torch.FloatTensor([Y])  
    out = net(torch.FloatTensor(X))  
    print("when x = {}, y = {}".format(X, out))
    

Net(
  (fc1): Linear(in_features=1, out_features=1, bias=False)
)
[Parameter containing:
tensor([[-0.9663]], requires_grad=True)]
Epoch 0 - loss: 15.731637954711914
Epoch 0 - loss: 60.434661865234375
Epoch 0 - loss: 112.55287170410156
Epoch 0 - loss: 118.76708221435547
Epoch 0 - loss: 52.32620620727539
Epoch 0 - loss: 0.25694647431373596
Epoch 1 - loss: 0.43226897716522217
Epoch 1 - loss: 4.123334884643555
Epoch 1 - loss: 11.148721694946289
Epoch 1 - loss: 14.791610717773438
Epoch 1 - loss: 8.354158401489258
Epoch 1 - loss: 0.340982049703598
Epoch 2 - loss: 0.04542175307869911
Epoch 2 - loss: 0.5302307605743408
Epoch 2 - loss: 1.516135811805725
Epoch 2 - loss: 2.0705621242523193
Epoch 2 - loss: 1.2019785642623901
Epoch 2 - loss: 0.05588885396718979
Epoch 3 - loss: 0.006225451361387968
Epoch 3 - loss: 0.07450179755687714
Epoch 3 - loss: 0.21439841389656067
Epoch 3 - loss: 0.29374828934669495
Epoch 3 - loss: 0.17103968560695648
Epoch 3 - loss: 0.008062991313636303
Epoch 4 - loss: 0.00088

In [4]:
for epoch in range(20):
    print(epoch)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


In [5]:
W = torch.tensor([1.0], requires_grad=True)
W = W*2
label = 1.0
loss = W*5 - label 
loss.backward()
W.grad

C:\Users\yangg\AppData\Local\Temp\ipykernel_34748\353321738.py:6: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:497.)
  W.grad
